# Layer-Wise Steering Propagation

Previous steering experiments injected directions at the final pooled layer. But *where* in the network does emotion information crystallize? This notebook builds emotion directions at each transformer layer and tests their steering effectiveness on the final classifier.

**Key insight:** If early-layer directions steer as effectively as late-layer directions, emotion encoding is distributed. If only late-layer directions work, emotion encoding is concentrated in the final layers — a more mechanistic understanding of where the model builds its emotion representations.

In [ ]:
from __future__ import annotations
import importlib.util, os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/pavannn16/speech-emotion-directions.git"
REPO_NAME = "speech-emotion-directions"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def running_in_colab():
    try: import google.colab; return True
    except ImportError: return False
IS_COLAB = running_in_colab()
print("Running in Colab:", IS_COLAB)

def run_command(cmd, cwd=None):
    print("+", " ".join(cmd)); subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)

def ensure_packages():
    pkgs = {"yaml":"pyyaml","pandas":"pandas","numpy":"numpy","matplotlib":"matplotlib",
            "seaborn":"seaborn","torch":"torch","transformers":"transformers","sklearn":"scikit-learn","tqdm":"tqdm"}
    missing = sorted({p for m,p in pkgs.items() if importlib.util.find_spec(m) is None})
    if missing: run_command([sys.executable,"-m","pip","install","-q",*missing])
    else: print("Required packages available.")

def find_local_project_root():
    cwd = Path.cwd().resolve()
    for c in [cwd, cwd.parent, cwd/"FinalProject"]:
        c = c.resolve()
        if (c/"configs"/"wav2vec.yaml").exists() and (c/"src").exists(): return c
    raise FileNotFoundError("Could not find project root.")

def clone_clean(cr):
    if cr.exists(): shutil.rmtree(cr)
    run_command(["git","clone","--depth","1",REPO_URL,str(cr)]); return cr

def prepare_roots():
    rt=Path("/content")/REPO_NAME; cl=Path("/content")/f"{REPO_NAME}-clean"
    if rt.exists() and (rt/".git").exists():
        try: run_command(["git","-C",str(rt),"fetch","origin"])
        except: pass
        st=subprocess.run(["git","-C",str(rt),"status","--short"],text=True,capture_output=True,check=True).stdout.splitlines()
        ab=subprocess.run(["git","-C",str(rt),"rev-list","--left-right","--count","HEAD...origin/main"],text=True,capture_output=True,check=False)
        a,b=(0,0) if ab.returncode!=0 or not ab.stdout.strip() else map(int,ab.stdout.strip().split())
        if not [l for l in st if l.strip()] and a==0:
            if b>0:
                try: run_command(["git","-C",str(rt),"pull","--ff-only","origin","main"]); cr=rt
                except: cr=clone_clean(cl)
            else: cr=rt
        else: cr=clone_clean(cl)
        return rt,cr
    elif rt.exists(): return rt,clone_clean(cl)
    else: run_command(["git","clone","--depth","1",REPO_URL,str(rt)]); return rt,rt

ensure_packages()
if IS_COLAB:
    from google.colab import drive; drive.mount('/content/drive',force_remount=False)
    PROJECT_ROOT,CODE_ROOT=prepare_roots()
else:
    PROJECT_ROOT=find_local_project_root(); CODE_ROOT=PROJECT_ROOT

os.chdir(PROJECT_ROOT)
for r in [str(CODE_ROOT),str(PROJECT_ROOT)]:
    while r in sys.path: sys.path.remove(r)
sys.path.insert(0,str(CODE_ROOT))
if str(PROJECT_ROOT)!=str(CODE_ROOT): sys.path.insert(1,str(PROJECT_ROOT))
for n in [n for n in sys.modules if n=="src" or n.startswith("src.")]: del sys.modules[n]
print("Project root:",PROJECT_ROOT); print("Code root:",CODE_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

experiment_name = 'wav2vec_ravdess_speaker_independent_v1'
local_checkpoint_dir = PROJECT_ROOT / 'artifacts' / 'checkpoints' / experiment_name
local_analysis_dir = PROJECT_ROOT / 'artifacts' / 'analysis' / experiment_name
local_output_dir = local_analysis_dir / 'layerwise_steering'
local_output_dir.mkdir(parents=True, exist_ok=True)

drive_run_root = Path('/content/drive/MyDrive') / 'speech-emotion-directions' / 'runs' / experiment_name if IS_COLAB else None
drive_checkpoint_dir = drive_run_root / 'checkpoint' if drive_run_root else None
drive_analysis_dir = drive_run_root / 'analysis' if drive_run_root else None
drive_output_dir = drive_analysis_dir / 'layerwise_steering' if drive_analysis_dir else None

checkpoint_dir = local_checkpoint_dir
if not (checkpoint_dir / 'model_state.pt').exists() and drive_checkpoint_dir and (drive_checkpoint_dir / 'model_state.pt').exists():
    checkpoint_dir = drive_checkpoint_dir
analysis_source_dir = local_analysis_dir
if not (analysis_source_dir / 'embedding_arrays.npz').exists() and drive_analysis_dir and (drive_analysis_dir / 'embedding_arrays.npz').exists():
    analysis_source_dir = drive_analysis_dir

from src.analysis.emotion_vectors import load_embedding_artifacts
from src.analysis.extract_embeddings import load_trained_checkpoint
from src.analysis.advanced_analysis import build_layerwise_directions, evaluate_layerwise_steering

artifacts = load_embedding_artifacts(analysis_source_dir)
label_names = list(artifacts.summary['label_names'])
reference_label = 'neutral'
reference_idx = label_names.index(reference_label)
label_ids = artifacts.true_label_ids
split_array = artifacts.metadata['split'].to_numpy()
train_mask = split_array == 'train'
test_mask = split_array == 'test'

checkpoint = load_trained_checkpoint(checkpoint_dir)
cw = checkpoint.model.classifier.weight.detach().cpu().numpy()
cb = checkpoint.model.classifier.bias.detach().cpu().numpy()

num_layers = artifacts.layer_embeddings.shape[1]
print(f'Total layers: {num_layers}')

## Build Layer-Wise Directions and Evaluate Steering

For each layer, build emotion directions from train centroids, then use those directions to steer the final-layer embeddings and measure the probability shift.

In [ ]:
# Build directions at every layer
layerwise_directions = build_layerwise_directions(
    layer_embeddings=artifacts.layer_embeddings,
    label_ids=label_ids,
    split_mask=train_mask,
    label_names=label_names,
    reference_idx=reference_idx,
)
print(f'Layerwise directions shape: {layerwise_directions.shape}')  # (num_layers, num_labels, hidden_size)

# Evaluate steering from each layer
target_ids = [i for i, n in enumerate(label_names) if i != reference_idx]
injection_layers = list(range(num_layers))

steering_df = evaluate_layerwise_steering(
    layer_embeddings=artifacts.layer_embeddings[test_mask],
    label_ids=label_ids[test_mask],
    layerwise_directions=layerwise_directions,
    classifier_weight=cw,
    classifier_bias=cb,
    label_names=label_names,
    reference_idx=reference_idx,
    target_label_ids=target_ids,
    injection_layers=injection_layers,
    alpha=0.5,
)
steering_df.to_csv(local_output_dir / 'layerwise_steering_results.csv', index=False)
display(steering_df.round(4))

In [ ]:
# Heatmap: steering effectiveness by injection layer and target emotion
pivot = steering_df.pivot(index='injection_layer', columns='target_label', values='mean_delta_target_prob')

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Steering Effectiveness by Injection Layer\n(Mean Delta Target Probability, alpha=0.5)')
ax.set_ylabel('Direction Source Layer')
ax.set_xlabel('Target Emotion')
plt.tight_layout()
plt.savefig(local_output_dir / 'layerwise_steering_heatmap.png', dpi=220)
plt.show()

In [ ]:
# Line plot: average steering delta across emotions per layer
avg_by_layer = steering_df.groupby('injection_layer')['mean_delta_target_prob'].mean().reset_index()
avg_by_layer.columns = ['layer', 'avg_delta']

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(avg_by_layer['layer'], avg_by_layer['avg_delta'], marker='o', linewidth=2, color='steelblue')
ax.fill_between(avg_by_layer['layer'], 0, avg_by_layer['avg_delta'], alpha=0.2, color='steelblue')
ax.set_xlabel('Direction Source Layer')
ax.set_ylabel('Average Delta Target Probability')
ax.set_title('Where Does Emotion Encoding Crystallize?\n(Average Steering Effectiveness by Layer)')
ax.set_xticks(range(num_layers))
ax.axhline(0, color='gray', linestyle='--')
plt.tight_layout()
plt.savefig(local_output_dir / 'layerwise_steering_average.png', dpi=220)
plt.show()

# Per-emotion line plots
fig, ax = plt.subplots(figsize=(10, 5))
for target_label in steering_df['target_label'].unique():
    subset = steering_df[steering_df['target_label'] == target_label]
    ax.plot(subset['injection_layer'], subset['mean_delta_target_prob'], marker='o', label=target_label)
ax.set_xlabel('Direction Source Layer')
ax.set_ylabel('Delta Target Probability')
ax.set_title('Layer-Wise Steering Effectiveness Per Emotion')
ax.legend()
ax.set_xticks(range(num_layers))
plt.tight_layout()
plt.savefig(local_output_dir / 'layerwise_steering_per_emotion.png', dpi=220)
plt.show()

## Multi-Alpha Sweep at Best vs Worst Layer

Compare steering curves at the most effective and least effective layers.

In [ ]:
from src.analysis.emotion_vectors import linear_classifier_probabilities

best_layer = int(avg_by_layer.loc[avg_by_layer['avg_delta'].idxmax(), 'layer'])
worst_layer = int(avg_by_layer.loc[avg_by_layer['avg_delta'].idxmin(), 'layer'])
print(f'Best steering layer: {best_layer}, Worst: {worst_layer}')

final_embeddings = artifacts.layer_embeddings[test_mask, num_layers - 1]
base_probs = linear_classifier_probabilities(final_embeddings, cw, cb)
alphas = np.arange(-1.0, 1.05, 0.1)

sweep_rows = []
for layer_idx in [best_layer, worst_layer]:
    for target_id in target_ids:
        direction = layerwise_directions[layer_idx, target_id]
        for alpha in alphas:
            steered = final_embeddings + alpha * direction[None, :]
            steered_probs = linear_classifier_probabilities(steered, cw, cb)
            sweep_rows.append({
                'source_layer': layer_idx,
                'target_label': label_names[target_id],
                'alpha': float(alpha),
                'mean_delta': float((steered_probs[:, target_id] - base_probs[:, target_id]).mean()),
            })

sweep_df = pd.DataFrame(sweep_rows)
sweep_df.to_csv(local_output_dir / 'best_worst_layer_alpha_sweep.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, layer_idx, title in zip(axes, [best_layer, worst_layer], [f'Best Layer ({best_layer})', f'Worst Layer ({worst_layer})']):
    subset = sweep_df[sweep_df['source_layer'] == layer_idx]
    for target_label in subset['target_label'].unique():
        t = subset[subset['target_label'] == target_label]
        ax.plot(t['alpha'], t['mean_delta'], marker='o', label=target_label, markersize=4)
    ax.set_xlabel('Alpha')
    ax.set_ylabel('Delta Target Probability')
    ax.set_title(title)
    ax.axhline(0, color='gray', linestyle='--')
    ax.legend(fontsize=8)

plt.suptitle('Steering Alpha Sweep: Best vs Worst Layer', y=1.02)
plt.tight_layout()
plt.savefig(local_output_dir / 'best_worst_layer_sweep.png', dpi=220)
plt.show()

In [ ]:
# --- Drive backup ---
if not IS_COLAB:
    print('Google Drive sync is intended for Colab runtimes. Skipping.')
elif drive_output_dir is None:
    print('Drive output directory is not configured.')
else:
    import shutil
    if drive_output_dir.exists(): shutil.rmtree(drive_output_dir)
    drive_output_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_output_dir, drive_output_dir)
    print('Backed up to:', drive_output_dir)

print('\nOutput files:')
for p in sorted(local_output_dir.iterdir()):
    print(f'  {p.name}')